<a href="https://colab.research.google.com/github/alnmuniz/pln-projeto-final/blob/main/projeto_final_pln_rascunho_andre.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Bloco 1: Instalação de pacotes e processamento do texto
!pip install -q pypdf langchain-text-splitters

# Baixando o PDF diretamente do Google Drive usando o ID do arquivo
print("Baixando o livro do Google Drive...")
!gdown 1vjb0YTkt8xuIsBKazk9tnW3v3h4F5Cl0 -O o_capital.pdf

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

# Caminho do arquivo que você fez upload no Colab
caminho_pdf = "o_capital.pdf"

def extrair_e_limpar_texto(caminho_arquivo):
    print("Lendo o PDF e extraindo o texto...")
    leitor = PdfReader(caminho_arquivo)
    texto_completo = ""

    for pagina in leitor.pages:
        texto_pagina = pagina.extract_text()
        if texto_pagina:
            texto_completo += texto_pagina

    # Limpeza básica: remove quebras de linha no meio de frases e espaços duplos
    texto_limpo = re.sub(r'(?<!\n)\n(?!\n)', ' ', texto_completo)
    texto_limpo = re.sub(r'\s+', ' ', texto_limpo).strip()
    print(f"Sucesso! Texto extraído com {len(texto_limpo)} caracteres.")
    return texto_limpo

def criar_chunks(texto):
    print("Dividindo o texto em chunks...")
    # O divisor tenta quebrar por parágrafos duplos, depois simples, depois pontos...
    divisor = RecursiveCharacterTextSplitter(
        chunk_size=600,   # Tamanho de cada pedaço de texto
        chunk_overlap=50, # Sobreposição para não perder o contexto entre um chunk e outro
        separators=["\n\n", "\n", ".", " ", ""]
    )
    chunks = divisor.split_text(texto)
    print(f"Total de {len(chunks)} chunks gerados.")
    return chunks

# Execução do pipeline da Etapa 1
try:
    texto_livro = extrair_e_limpar_texto(caminho_pdf)
    chunks_livro = criar_chunks(texto_livro)

    # Exibindo uma amostra para verificação
    print("\n--- Amostra do 10º chunk gerado ---")
    print(chunks_livro[9])
except FileNotFoundError:
    print("ERRO: O arquivo 'o_capital.pdf' não foi encontrado. Verifique se o upload terminou no painel lateral do Colab.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 6.0 MB/s eta 0:00:00
Baixando o livro do Google Drive...
Downloading...
From: https://drive.google.com/uc?id=1vjb0YTkt8xuIsBKazk9tnW3v3h4F5Cl0
To: /content/o_capital.pdf
100% 8.72M/8.72M [00:00<00:00, 22.5MB/s]
Lendo o PDF e extraindo o texto...
Sucesso! Texto extraído com 2576954 caracteres.
Dividindo o texto em chunks...
Total de 5358 chunks gerados.

--- Amostra do 10º chunk gerado ---
. Quem espera que este livro comece pelo exame do capital, prepare-se para um anticlímax: Marx examina antes de tudo a mercadoria e sua formação, pois o capitalismo continua a ser, mesmo em sua fase amplamente financeirizada, um modo de produção de mercadorias


In [ ]:
# Bloco 2: Instalação, Vetorização (E5) e Indexação (Qdrant)
!pip install -q qdrant-client sentence-transformers

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer
import uuid

# 1. Inicializar o cliente do Qdrant em memória (perfeito e rápido para o Colab)
print("Inicializando o banco vetorial Qdrant...")
qdrant = QdrantClient(location=":memory:")

# Nome da nossa coleção no banco
NOME_COLECAO = "o_capital_marx"

# 2. Inicializar o modelo de embeddings E5
print("Carregando o modelo de embeddings (o download pode levar um minutinho)...")
modelo_embedding = SentenceTransformer("intfloat/multilingual-e5-base")

# A dimensão do vetor gerado pelo e5-base é 768
DIMENSAO_VETOR = modelo_embedding.get_sentence_embedding_dimension()

# 3. Criar a coleção no Qdrant
qdrant.create_collection(
    collection_name=NOME_COLECAO,
    vectors_config=VectorParams(size=DIMENSAO_VETOR, distance=Distance.COSINE),
)
print(f"Coleção '{NOME_COLECAO}' criada com sucesso!")

# 4. Função para vetorizar e indexar os chunks
def indexar_chunks(chunks, cliente_qdrant, modelo, nome_colecao):
    print(f"Iniciando a vetorização de {len(chunks)} chunks... Isso vai levar um tempinho.")
    pontos = []

    for i, chunk in enumerate(chunks):
        # O modelo E5 exige o prefixo "passage: " para os documentos armazenados
        texto_para_vetorizar = f"passage: {chunk}"

        # Gerar o embedding (vetor)
        vetor = modelo.encode(texto_para_vetorizar).tolist()

        # Criar a estrutura do ponto para o Qdrant (ID único, Vetor e o Payload com o texto original)
        ponto = PointStruct(
            id=str(uuid.uuid4()),
            vector=vetor,
            payload={"texto": chunk, "id_chunk": i} # Guardamos o texto original sem o prefixo para leitura depois
        )
        pontos.append(ponto)

        # Enviando ao Qdrant em lotes de 100 para não estourar a RAM do Colab
        if len(pontos) >= 100 or i == len(chunks) - 1:
            cliente_qdrant.upsert(
                collection_name=nome_colecao,
                points=pontos
            )
            pontos = [] # Limpa a lista para o próximo lote

            # Print de progresso a cada 500 chunks para não poluir muito a tela
            if (i + 1) % 500 == 0 or i == len(chunks) - 1:
                print(f"-> Indexados {i+1}/{len(chunks)} chunks...")

    print("Indexação concluída com sucesso!")

# Executar a vetorização (utiliza a variável chunks_livro do Bloco 1)
if 'chunks_livro' in locals() and len(chunks_livro) > 0:
    indexar_chunks(chunks_livro, qdrant, modelo_embedding, NOME_COLECAO)

    # Conferência final
    info = qdrant.get_collection(NOME_COLECAO)
    print(f"\nStatus final do Qdrant: {info.points_count} vetores armazenados perfeitamente.")
else:
    print("ERRO: A variável 'chunks_livro' não foi encontrada. Certifique-se de rodar o Bloco 1 com sucesso antes deste.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 7.6 MB/s eta 0:00:00
Inicializando o banco vetorial Qdrant...
Carregando o modelo de embeddings (o download pode levar um minutinho)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Coleção 'o_capital_marx' criada com sucesso!
Iniciando a vetorização de 5358 chunks... Isso vai levar um tempinho.
-> Indexados 500/5358 chunks...
-> Indexados 1000/5358 chunks...


In [ ]:
# Bloco 3: Pipeline de Recuperação (Buscador no Qdrant)

def buscar_contexto(pergunta, cliente_qdrant, modelo, nome_colecao, top_k=3):
    print(f"Buscando as {top_k} passagens mais relevantes no livro...")

    # 1. Aplicar o prefixo obrigatório do modelo E5 para perguntas
    pergunta_formatada = f"query: {pergunta}"

    # 2. Transformar a pergunta em vetor (embedding)
    vetor_pergunta = modelo.encode(pergunta_formatada).tolist()

    # 3. Fazer a busca de similaridade (Cosseno) no Qdrant
    resultados = cliente_qdrant.search(
        collection_name=nome_colecao,
        query_vector=vetor_pergunta,
        limit=top_k
    )

    # 4. Extrair e retornar apenas os textos recuperados
    contextos_recuperados = []
    print("\nResultados encontrados:")
    for i, res in enumerate(resultados):
        texto = res.payload['texto']
        score = res.score
        contextos_recuperados.append(texto)

        # Imprime um resumo do que encontrou para conferência
        print(f"\n--- Top {i+1} (Similaridade: {score:.4f}) ---")
        print(f"{texto[:250]}...") # Mostra os primeiros 250 caracteres

    return contextos_recuperados

# ==========================================
# TESTE DO BUSCADOR
# ==========================================
pergunta_teste = "O que é a mais-valia e como o capitalista lucra com ela?"
print(f"Pergunta de teste: '{pergunta_teste}'\n")

# Vamos buscar os 4 chunks mais relevantes
contextos = buscar_contexto(pergunta_teste, qdrant, modelo_embedding, NOME_COLECAO, top_k=4)

In [ ]:
# Bloco 4: Geração Aumentada (RAG) com o Qwen 2.5
!pip install -q accelerate transformers torch

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Definindo o modelo escolhido
NOME_LLM = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Carregando o modelo {NOME_LLM} (isso vai baixar cerca de 3GB)...")

# Carregando o Tokenizador e o Modelo na GPU (float16 economiza muita memória)
tokenizer_llm = AutoTokenizer.from_pretrained(NOME_LLM)
modelo_llm = AutoModelForCausalLM.from_pretrained(
    NOME_LLM,
    torch_dtype=torch.float16,
    device_map="auto" # Manda automaticamente para a GPU se disponível
)
print("LLM carregado com sucesso na GPU!")

def gerar_resposta_qwen(pergunta, contextos, modelo, tokenizer):
    # 1. Juntar os blocos de texto recuperados do Qdrant em um único texto contínuo
    contexto_completo = "\n\n---\n\n".join(contextos)

    # 2. Montar a instrução do sistema e do usuário (Engenharia de Prompt)
    mensagens = [
        {"role": "system", "content": "Você é um assistente acadêmico especialista na obra 'O Capital' de Karl Marx. Responda à pergunta do usuário utilizando ÚNICA E EXCLUSIVAMENTE as informações fornecidas no contexto abaixo. Se a resposta não estiver no contexto, diga que não sabe com base nos trechos fornecidos."},
        {"role": "user", "content": f"CONTEXTO RECUPERADO:\n{contexto_completo}\n\nPERGUNTA: {pergunta}"}
    ]

    # 3. Preparar o prompt no formato de chat que o Qwen entende
    texto_prompt = tokenizer.apply_chat_template(
        mensagens,
        tokenize=False,
        add_generation_prompt=True
    )

    # 4. Tokenizar e enviar os dados para a GPU
    inputs = tokenizer([texto_prompt], return_tensors="pt").to(modelo.device)

    # 5. Gerar a resposta
    print("\nO Qwen está analisando o texto e formulando a resposta...")
    with torch.no_grad():
        outputs = modelo.generate(
            **inputs,
            max_new_tokens=512, # Limite de tamanho da resposta
            temperature=0.3,    # Temperatura baixa (0.3) reduz a criatividade e foca nos fatos do texto (evita alucinações)
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # 6. Decodificar apenas a parte gerada (removendo o prompt original que enviamos)
    tamanho_input = inputs.input_ids.shape[1]
    resposta_gerada = tokenizer.decode(outputs[0][tamanho_input:], skip_special_tokens=True)

    return resposta_gerada

# ==========================================
# TESTE DO RAG COMPLETO
# ==========================================
# Usaremos a variável 'contextos' e 'pergunta_teste' que criamos no Bloco 3
if 'contextos' in locals() and len(contextos) > 0:
    resposta_final = gerar_resposta_qwen(pergunta_teste, contextos, modelo_llm, tokenizer_llm)
    print("\n" + "="*60)
    print("🤖 RESPOSTA DO RAG:")
    print("="*60)
    print(resposta_final)
else:
    print("Aguardando o Bloco 3 para rodar o teste completo.")

In [ ]:
# Bloco 5: Criando e rodando a Interface Streamlit
!pip install -q streamlit
!npm install localtunnel

%%writefile app.py
import streamlit as st
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

# 1. Configuração da Página
st.set_page_config(page_title="Chatbot O Capital - RAG", page_icon="📕")
st.title("🤖 Assistente de 'O Capital' (Karl Marx)")
st.write("Faça perguntas sobre a obra com base no texto indexado.")

# 2. Carregamento dos Modelos (com cache para não recarregar a cada interação)
@st.cache_resource
def carregar_recursos():
    # Inicializa o Qdrant apontando para o diretório salvo em disco
    qdrant = QdrantClient(path="./qdrant_marx_db")

    # Carrega Embeddings
    modelo_emb = SentenceTransformer("intfloat/multilingual-e5-base")

    # Carrega LLM
    nome_llm = "Qwen/Qwen2.5-1.5B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(nome_llm)
    modelo_llm = AutoModelForCausalLM.from_pretrained(
        nome_llm, torch_dtype=torch.float16, device_map="auto"
    )

    return qdrant, modelo_emb, modelo_llm, tokenizer

qdrant, modelo_emb, modelo_llm, tokenizer = carregar_recursos()
NOME_COLECAO = "o_capital_marx"

# 3. Histórico do Chat
if "mensagens" not in st.session_state:
    st.session_state.mensagens = []

for msg in st.session_state.mensagens:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# 4. Input do Usuário
pergunta = st.chat_input("Pergunte algo sobre a obra...")

if pergunta:
    # Exibe pergunta do usuário
    with st.chat_message("user"):
        st.markdown(pergunta)
    st.session_state.mensagens.append({"role": "user", "content": pergunta})

    with st.chat_message("assistant"):
        placeholder = st.empty()
        placeholder.markdown("Buscando passagens relevantes e formulando resposta...")

        # Recuperação (Retriever)
        pergunta_formatada = f"query: {pergunta}"
        vetor_pergunta = modelo_emb.encode(pergunta_formatada).tolist()
        resultados = qdrant.search(
            collection_name=NOME_COLECAO, query_vector=vetor_pergunta, limit=4
        )
        contextos = [res.payload['texto'] for res in resultados]
        contexto_completo = "\n\n---\n\n".join(contextos)

        # Geração (LLM)
        mensagens_llm = [
            {"role": "system", "content": "Você é um assistente acadêmico especialista em 'O Capital'. Responda usando EXCLUSIVAMENTE o contexto fornecido."},
            {"role": "user", "content": f"CONTEXTO:\n{contexto_completo}\n\nPERGUNTA: {pergunta}"}
        ]
        texto_prompt = tokenizer.apply_chat_template(mensagens_llm, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([texto_prompt], return_tensors="pt").to(modelo_llm.device)

        with torch.no_grad():
            outputs = modelo_llm.generate(**inputs, max_new_tokens=512, temperature=0.3, do_sample=True, pad_token_id=tokenizer.eos_token_id)

        tamanho_input = inputs.input_ids.shape[1]
        resposta = tokenizer.decode(outputs[0][tamanho_input:], skip_special_tokens=True)

        # Mostra resposta e fontes
        resposta_final = f"{resposta}\n\n**Trechos utilizados como base:**\n"
        for i, txt in enumerate(contextos):
            resposta_final += f"- *{txt[:150]}...*\n"

        placeholder.markdown(resposta_final)
        st.session_state.mensagens.append({"role": "assistant", "content": resposta_final})

In [ ]:
import urllib
print("Senha/IP para acessar o site:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())
!streamlit run app.py & npx localtunnel --port 8501